# EMA Crossover + Support Touch Scanner
### Multi-Timeframe | Multi-EMA Pair | SL & Target Analysis

**Strategy Logic:**
1. **Cross Event:** Fast EMA crosses **above** Slow EMA (bullish crossover)
2. **Touch Condition** (all candles after cross; invalidated if Fast EMA drops below Slow EMA):
   - `Low ≤ Fast EMA` — low entered the EMA support zone
   - `Close > Fast EMA` — closed back above Fast EMA (support held)
   - `Close > Open` — bullish / green candle
3. **Touch Zone Labels:**
   - `Between Slow & Fast EMA` — Low is between the two EMAs
   - `At/Below Slow EMA` — Low touched or pierced Slow EMA


In [6]:
# ============================================================
#  CONFIG  —  ONLY EDIT THIS CELL
# ============================================================

# Stocks (must match the prefix in your CSV filenames)
STOCKS = [
    "RELIANCE",
    "TCS",
    "HDFCBANK",
    "INFY",
    "ICICIBANK",
]

# ── FILE NAME MAP ────────────────────────────────────────────────────────
# Maps interval label → filename pattern  ("{symbol}" replaced at runtime)
FILE_MAP = {
    "1d": "data/1d/{symbol}_historical.csv",
    "1h": "data/1h/{symbol}_historical.csv",
}

# ── SETUPS ───────────────────────────────────────────────────────────────
# Each triplet = [interval, fast_ema, slow_ema]
SETUPS = [
    ["1d",  21, 50],
    ["1d",  10, 20],
    ["1h",  21, 50],
    ["1h",  10, 20],
]

# Folder where CSV files are stored ("." = same folder as notebook)
DATA_FOLDER = "."

# ============================================================
#  END CONFIG
# ============================================================


In [7]:
# ── IMPORTS + SETUP ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows",    300)
pd.set_option("display.width",       300)
pd.set_option("display.float_format", "{:.2f}".format)

# Validate: every interval in SETUPS must exist in FILE_MAP
missing = [s[0] for s in SETUPS if s[0] not in FILE_MAP]
if missing:
    raise ValueError(f"Interval(s) {missing} not found in FILE_MAP. Add them to FILE_MAP first.")

TIMEFRAMES = list(dict.fromkeys([s[0] for s in SETUPS]))

print("=" * 60)
print("  EMA Crossover Scanner — Config Summary")
print("=" * 60)
print(f"  Stocks     : {STOCKS}")
print(f"  Timeframes : {TIMEFRAMES}")
print(f"  Setups     :")
for s in SETUPS:
    print(f"               [{s[0]}]  EMA{s[1]} / EMA{s[2]}")
print()
print("  File mapping:")
for interval, pattern in FILE_MAP.items():
    example = pattern.replace("{symbol}", STOCKS[0])
    print(f"               {interval:6s} → {example}")
print(f"  Data folder: {DATA_FOLDER}")
print("=" * 60)


  EMA Crossover Scanner — Config Summary
  Stocks     : ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK']
  Timeframes : ['1d', '1h']
  Setups     :
               [1d]  EMA21 / EMA50
               [1d]  EMA10 / EMA20
               [1h]  EMA21 / EMA50
               [1h]  EMA10 / EMA20

  File mapping:
               1d     → data/1d/RELIANCE_historical.csv
               1h     → data/1h/RELIANCE_historical.csv
  Data folder: .


In [8]:
import sys
print(sys.executable)

c:\Users\User\AppData\Local\Programs\Python\Python311\python.exe


In [9]:
# ── LOAD CSV → DataFrame ────────────────────────────────────────────────────

def load_csv(symbol: str, interval: str) -> pd.DataFrame:
    """
    Load OHLCV data from a local CSV file.
    Handles both 'datetime' (string) and 'timestamp' (Unix epoch) columns.
    """
    pattern  = FILE_MAP[interval]
    filename = pattern.replace("{symbol}", symbol)
    filepath = os.path.join(DATA_FOLDER, filename)

    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"File not found: '{filepath}'\n"
            f"  Expected pattern for [{interval}]: {filename}\n"
            f"  Check DATA_FOLDER and FILE_MAP in CONFIG cell."
        )

    df = pd.read_csv(filepath)
    df.columns = [c.strip().lower() for c in df.columns]

    if "datetime" in df.columns:
        df["Date"] = pd.to_datetime(df["datetime"])
    elif "timestamp" in df.columns:
        df["Date"] = (
            pd.to_datetime(df["timestamp"], unit="s")
            + pd.Timedelta(hours=5, minutes=30)   # UTC → IST
        )
    else:
        raise ValueError(f"CSV '{filename}' has no 'datetime' or 'timestamp' column.")

    df = df.set_index("Date").sort_index()
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)

    df = df.rename(columns={"open": "Open", "high": "High",
                             "low":  "Low",  "close": "Close", "volume": "Volume"})
    cols = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in df.columns]
    return df[cols].dropna()


# ── Quick sanity check ───────────────────────────────────────────────────────
print("Checking CSV files ...")
print("-" * 55)
all_ok = True
for interval in TIMEFRAMES:
    for symbol in STOCKS:
        try:
            df = load_csv(symbol, interval)
            print(f"  ✓ {symbol:12s} [{interval}]  {len(df):5d} rows  "
                  f"({df.index[0].date()} → {df.index[-1].date()})")
        except Exception as e:
            print(f"  ✗ {symbol:12s} [{interval}]  ERROR: {e}")
            all_ok = False
print("-" * 55)
print("All files OK." if all_ok else "Fix errors above before running the scanner.")


Checking CSV files ...
-------------------------------------------------------
  ✓ RELIANCE     [1d]    582 rows  (2024-01-02 → 2026-05-08)
  ✓ TCS          [1d]    582 rows  (2024-01-02 → 2026-05-08)
  ✓ HDFCBANK     [1d]    582 rows  (2024-01-02 → 2026-05-08)
  ✓ INFY         [1d]    582 rows  (2024-01-02 → 2026-05-08)
  ✓ ICICIBANK    [1d]    582 rows  (2024-01-02 → 2026-05-08)
  ✓ RELIANCE     [1h]    595 rows  (2026-01-01 → 2026-05-08)
  ✓ TCS          [1h]    595 rows  (2026-01-01 → 2026-05-08)
  ✓ HDFCBANK     [1h]    595 rows  (2026-01-01 → 2026-05-08)
  ✓ INFY         [1h]    595 rows  (2026-01-01 → 2026-05-08)
  ✓ ICICIBANK    [1h]    595 rows  (2026-01-01 → 2026-05-08)
-------------------------------------------------------
All files OK.


In [10]:
# ── EMA CALCULATION + SCANNER LOGIC ────────────────────────────────────────

def add_emas(df: pd.DataFrame, fast: int, slow: int) -> pd.DataFrame:
    """Pine Script ta.ema() equivalent: ewm(span=N, adjust=False)."""
    df = df.copy()
    df[f"EMA{fast}"] = df["Close"].ewm(span=fast, adjust=False).mean()
    df[f"EMA{slow}"] = df["Close"].ewm(span=slow, adjust=False).mean()
    return df


def bullish_cross_indices(df: pd.DataFrame, fast: int, slow: int) -> list:
    """Bar positions where Fast EMA crosses above Slow EMA."""
    above = df[f"EMA{fast}"] > df[f"EMA{slow}"]
    cross = above & ~above.shift(1, fill_value=False)
    cross.iloc[0] = False
    return list(np.where(cross.values)[0])


def touch_zone(low: float, ema_fast: float, ema_slow: float) -> str:
    if low <= ema_slow:
        return "At/Below Slow EMA"
    return "Between Slow & Fast EMA"


def scan(stock: str, interval: str, fast: int, slow: int,
         df: pd.DataFrame) -> list:
    """Scan one stock/timeframe/EMA-pair combination."""
    df   = add_emas(df, fast, slow)
    rows = []

    for cidx in bullish_cross_indices(df, fast, slow):
        cross_ts    = df.index[cidx]
        cross_time  = str(cross_ts)
        cross_close = float(df["Close"].iloc[cidx])
        ef_cross    = float(df[f"EMA{fast}"].iloc[cidx])
        es_cross    = float(df[f"EMA{slow}"].iloc[cidx])

        for i in range(cidx + 1, len(df)):
            low    = float(df["Low"].iloc[i])
            close  = float(df["Close"].iloc[i])
            open_  = float(df["Open"].iloc[i])
            high   = float(df["High"].iloc[i])
            ef     = float(df[f"EMA{fast}"].iloc[i])
            es     = float(df[f"EMA{slow}"].iloc[i])
            touch_ts = df.index[i]

            if ef <= es:   # cross invalidated
                break

            if low <= ef and close > ef and close > open_:
                rows.append({
                    "Stock":               stock,
                    "Timeframe":           interval,
                    "EMA Pair":            f"EMA{fast}/EMA{slow}",
                    "Cross Time":          cross_time,
                    "Cross Close":         round(cross_close, 2),
                    f"EMA{fast} @ Cross":  round(ef_cross, 2),
                    f"EMA{slow} @ Cross":  round(es_cross, 2),
                    "Touch Time":          str(touch_ts),
                    "Touch Open":          round(open_, 2),
                    "Touch High":          round(high, 2),
                    "Touch Low":           round(low, 2),
                    "Touch Close":         round(close, 2),
                    f"EMA{fast} @ Touch":  round(ef, 2),
                    f"EMA{slow} @ Touch":  round(es, 2),
                    "Touch Zone":          touch_zone(low, ef, es),
                    "Candles After Cross": i - cidx,
                })
    return rows


print("Scanner functions loaded.")


Scanner functions loaded.


In [11]:
# ── RUN SCANNER — ALL SETUPS × ALL STOCKS ───────────────────────────────────

all_signals = []

print(f"Running {len(SETUPS)} setup(s) × {len(STOCKS)} stock(s) = "
      f"{len(SETUPS)*len(STOCKS)} scans")
print("-" * 60)

for setup in SETUPS:
    interval, fast, slow = setup
    label = f"[{interval}] EMA{fast}/EMA{slow}"
    for stock in STOCKS:
        tag = f"{stock:12s} {label}"
        try:
            df  = load_csv(stock, interval)
            sig = scan(stock, interval, fast, slow, df)
            print(f"  ✓ {tag:35s} → {len(sig):3d} signal(s)")
            all_signals.extend(sig)
        except Exception as e:
            print(f"  ✗ {tag:35s} → ERROR: {e}")

signals = pd.DataFrame(all_signals) if all_signals else pd.DataFrame()

if not signals.empty:
    signals = signals.sort_values(
        ["Stock", "Timeframe", "EMA Pair", "Cross Time", "Touch Time"]
    ).reset_index(drop=True)
    signals.index += 1

print("-" * 60)
print(f"Total signals found : {len(signals)}")


Running 4 setup(s) × 5 stock(s) = 20 scans
------------------------------------------------------------
  ✓ RELIANCE     [1d] EMA21/EMA50       →  24 signal(s)
  ✓ TCS          [1d] EMA21/EMA50       →  19 signal(s)
  ✓ HDFCBANK     [1d] EMA21/EMA50       →  40 signal(s)
  ✓ INFY         [1d] EMA21/EMA50       →  22 signal(s)
  ✓ ICICIBANK    [1d] EMA21/EMA50       →  36 signal(s)
  ✓ RELIANCE     [1d] EMA10/EMA20       →  29 signal(s)
  ✓ TCS          [1d] EMA10/EMA20       →  35 signal(s)
  ✓ HDFCBANK     [1d] EMA10/EMA20       →  56 signal(s)
  ✓ INFY         [1d] EMA10/EMA20       →  42 signal(s)
  ✓ ICICIBANK    [1d] EMA10/EMA20       →  67 signal(s)
  ✓ RELIANCE     [1h] EMA21/EMA50       →  17 signal(s)
  ✓ TCS          [1h] EMA21/EMA50       →   9 signal(s)
  ✓ HDFCBANK     [1h] EMA21/EMA50       →   5 signal(s)
  ✓ INFY         [1h] EMA21/EMA50       →  17 signal(s)
  ✓ ICICIBANK    [1h] EMA21/EMA50       →  13 signal(s)
  ✓ RELIANCE     [1h] EMA10/EMA20       →  31 signal(s)


In [12]:
# ── FULL SIGNAL TABLE ───────────────────────────────────────────────────────
if not signals.empty:
    print(f"All {len(signals)} qualifying touch candles:\n")
    display(signals)
else:
    print("No signals found.")


All 559 qualifying touch candles:



,Stock,Timeframe,EMA Pair,Cross Time,Cross Close,EMA21 @ Cross,EMA50 @ Cross,Touch Time,Touch Open,Touch High,Touch Low,Touch Close,EMA21 @ Touch,EMA50 @ Touch,Touch Zone,Candles After Cross,EMA10 @ Cross,EMA20 @ Cross,EMA10 @ Touch,EMA20 @ Touch
1,HDFCBANK,1d,EMA10/EMA20,2024-03-13 00:00:00,730.20,NaN,NaN,2024-03-15 00:00:00,725.00,729.65,721.12,726.33,NaN,NaN,Between Slow & Fast EMA,2,720.60,719.68,722.70,721.00
2,HDFCBANK,1d,EMA10/EMA20,2024-03-13 00:00:00,730.20,NaN,NaN,2024-03-19 00:00:00,717.75,726.20,717.75,724.67,NaN,NaN,At/Below Slow EMA,4,720.60,719.68,723.11,721.53
3,HDFCBANK,1d,EMA10/EMA20,2024-03-13 00:00:00,730.20,NaN,NaN,2024-03-21 00:00:00,721.30,725.62,719.15,722.88,NaN,NaN,At/Below Slow EMA,6,720.60,719.68,721.94,721.14
4,HDFCBANK,1d,EMA10/EMA20,2024-03-28 00:00:00,723.95,NaN,NaN,2024-04-19 00:00:00,743.27,767.48,740.12,765.65,NaN,NaN,At/Below Slow EMA,13,720.89,720.70,755.62,748.03
5,HDFCBANK,1d,EMA10/EMA20,2024-03-28 00:00:00,723.95,NaN,NaN,2024-04-29 00:00:00,757.50,767.23,753.27,764.75,NaN,NaN,Between Slow & Fast EMA,19,720.89,720.70,757.05,752.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
555,TCS,1h,EMA21/EMA50,2026-04-01 13:15:00,2415.70,2406.05,2405.95,2026-04-02 11:15:00,2405.30,2414.70,2403.10,2406.60,2406.59,2406.25,At/Below Slow EMA,5,NaN,NaN,NaN,NaN
556,TCS,1h,EMA21/EMA50,2026-04-01 13:15:00,2415.70,2406.05,2405.95,2026-04-02 12:15:00,2407.10,2420.40,2403.00,2419.80,2407.79,2406.78,At/Below Slow EMA,6,NaN,NaN,NaN,NaN
557,TCS,1h,EMA21/EMA50,2026-04-01 13:15:00,2415.70,2406.05,2405.95,2026-04-07 09:15:00,2461.00,2500.00,2448.00,2499.80,2448.15,2428.44,Between Slow & Fast EMA,17,NaN,NaN,NaN,NaN
558,TCS,1h,EMA21/EMA50,2026-04-01 13:15:00,2415.70,2406.05,2405.95,2026-04-15 09:15:00,2501.90,2553.20,2496.60,2549.30,2510.56,2501.15,At/Below Slow EMA,52,NaN,NaN,NaN,NaN


In [13]:
# ── EXPORT SIGNALS ──────────────────────────────────────────────────────────
signals_dir = "signals"
os.makedirs(signals_dir, exist_ok=True)

if not signals.empty:
    out = os.path.join(signals_dir, "signals_all.csv")
    signals.to_csv(out, index=False)
    print(f"Full signal table saved → {out}")
    for (tf, pair), grp in signals.groupby(["Timeframe", "EMA Pair"]):
        fname = f"signals_{tf}_{pair.replace('/', '_')}.csv"
        fpath = os.path.join(signals_dir, fname)
        grp.to_csv(fpath, index=False)
        print(f"  Saved → {fpath}  ({len(grp)} rows)")
    print(f"\nAll signal files saved in → '{signals_dir}/'")


Full signal table saved → signals\signals_all.csv
  Saved → signals\signals_1d_EMA10_EMA20.csv  (229 rows)
  Saved → signals\signals_1d_EMA21_EMA50.csv  (141 rows)
  Saved → signals\signals_1h_EMA10_EMA20.csv  (128 rows)
  Saved → signals\signals_1h_EMA21_EMA50.csv  (61 rows)

All signal files saved in → 'signals/'


## SL & Target Analysis

In [14]:
# ── SL ENGINE ──────────────────────────────────────────────────────────────

def candles_to_duration(n_candles: int, interval: str) -> str:
    if interval in ("60", "1h"):
        return f"{n_candles}h"
    elif interval == "1d":
        return f"{n_candles}d"
    return f"{n_candles} candles"


def add_sl_and_outcome(signals: pd.DataFrame,
                       stock_data: dict) -> pd.DataFrame:
    """
    SL Price     = Touch Low
    SL Hit       = subsequent candle Low < SL Price
    Max High     = highest High from touch+1 until SL hit (or end of data)
    Max Upside % = (Max High - Touch Close) / Touch Close * 100
    """
    results = []
    for _, row in signals.iterrows():
        stock    = row["Stock"]
        interval = row["Timeframe"]
        sl_price = row["Touch Low"]
        t_close  = row["Touch Close"]
        touch_ts = pd.Timestamp(row["Touch Time"])

        df = stock_data.get((stock, interval))
        if df is None:
            results.append({**row, "SL Price": sl_price, "SL Hit": "N/A",
                            "SL Hit Time": None, "Max High Before SL": None,
                            "Max Upside %": None, "Trade Duration": None})
            continue

        try:
            touch_idx = df.index.get_loc(touch_ts)
        except KeyError:
            touch_idx = df.index.searchsorted(touch_ts)

        sl_hit = False; sl_hit_time = None; sl_hit_idx = None
        max_high = float("-inf")

        for j in range(touch_idx + 1, len(df)):
            candle_low  = float(df["Low"].iloc[j])
            candle_high = float(df["High"].iloc[j])
            max_high = max(max_high, candle_high)
            if candle_low < sl_price:
                sl_hit = True; sl_hit_time = df.index[j]; sl_hit_idx = j
                break

        if max_high == float("-inf"):
            max_high = None
        max_upside = (round((max_high - t_close) / t_close * 100, 2)
                      if max_high is not None else None)

        n_candles = (sl_hit_idx - touch_idx if sl_hit and sl_hit_idx is not None
                     else len(df) - 1 - touch_idx)
        duration  = candles_to_duration(n_candles, interval) if n_candles >= 0 else None

        results.append({
            **row,
            "SL Price":           round(sl_price, 2),
            "SL Hit":             "Yes" if sl_hit else "No",
            "SL Hit Time":        str(sl_hit_time) if sl_hit_time else "Not Hit",
            "Max High Before SL": round(max_high, 2) if max_high else None,
            "Max Upside %":       max_upside,
            "Trade Duration":     duration,
        })
    return pd.DataFrame(results)


print("SL engine loaded.")


SL engine loaded.


In [15]:
# ── PRE-LOAD STOCK DATA + APPLY SL ─────────────────────────────────────────

stock_data = {}
print("Pre-loading stock data ...")

for setup in SETUPS:
    interval, fast, slow = setup
    for stock in STOCKS:
        key = (stock, interval)
        if key not in stock_data:
            try:
                df = load_csv(stock, interval)
                df = add_emas(df, fast, slow)
                stock_data[key] = df
                print(f"  ✓ {stock} [{interval}]")
            except Exception as e:
                print(f"  ✗ {stock} [{interval}] ERROR: {e}")
                stock_data[key] = None

print()
print("Applying SL + outcome analysis ...")
signals_with_sl = add_sl_and_outcome(signals, stock_data)

sl_hit_count = (signals_with_sl["SL Hit"] == "Yes").sum()
total        = len(signals_with_sl)
print(f"Done.")
print(f"  Total signals : {total}")
print(f"  SL Hit        : {sl_hit_count}  ({sl_hit_count/total*100:.1f}%)")
print(f"  SL Not Hit    : {total - sl_hit_count}  ({(total-sl_hit_count)/total*100:.1f}%)")


Pre-loading stock data ...
  ✓ RELIANCE [1d]
  ✓ TCS [1d]
  ✓ HDFCBANK [1d]
  ✓ INFY [1d]
  ✓ ICICIBANK [1d]
  ✓ RELIANCE [1h]
  ✓ TCS [1h]
  ✓ HDFCBANK [1h]
  ✓ INFY [1h]
  ✓ ICICIBANK [1h]

Applying SL + outcome analysis ...
Done.
  Total signals : 559
  SL Hit        : 546  (97.7%)
  SL Not Hit    : 13  (2.3%)


In [16]:
# ── FULL SIGNAL TABLE WITH SL + OUTCOMES ───────────────────────────────────
display(signals_with_sl)


,Stock,Timeframe,EMA Pair,Cross Time,Cross Close,EMA21 @ Cross,EMA50 @ Cross,Touch Time,Touch Open,Touch High,Touch Low,Touch Close,EMA21 @ Touch,EMA50 @ Touch,Touch Zone,Candles After Cross,EMA10 @ Cross,EMA20 @ Cross,EMA10 @ Touch,EMA20 @ Touch,SL Price,SL Hit,SL Hit Time,Max High Before SL,Max Upside %,Trade Duration
0,HDFCBANK,1d,EMA10/EMA20,2024-03-13 00:00:00,730.20,NaN,NaN,2024-03-15 00:00:00,725.00,729.65,721.12,726.33,NaN,NaN,Between Slow & Fast EMA,2,720.60,719.68,722.70,721.00,721.12,Yes,2024-03-18 00:00:00,728.00,0.23,1d
1,HDFCBANK,1d,EMA10/EMA20,2024-03-13 00:00:00,730.20,NaN,NaN,2024-03-19 00:00:00,717.75,726.20,717.75,724.67,NaN,NaN,At/Below Slow EMA,4,720.60,719.68,723.11,721.53,717.75,Yes,2024-03-20 00:00:00,725.83,0.16,1d
2,HDFCBANK,1d,EMA10/EMA20,2024-03-13 00:00:00,730.20,NaN,NaN,2024-03-21 00:00:00,721.30,725.62,719.15,722.88,NaN,NaN,At/Below Slow EMA,6,720.60,719.68,721.94,721.14,719.15,Yes,2024-03-22 00:00:00,725.38,0.35,1d
3,HDFCBANK,1d,EMA10/EMA20,2024-03-28 00:00:00,723.95,NaN,NaN,2024-04-19 00:00:00,743.27,767.48,740.12,765.65,NaN,NaN,At/Below Slow EMA,13,720.89,720.70,755.62,748.03,740.12,Yes,2024-05-09 00:00:00,778.70,1.70,13d
4,HDFCBANK,1d,EMA10/EMA20,2024-03-28 00:00:00,723.95,NaN,NaN,2024-04-29 00:00:00,757.50,767.23,753.27,764.75,NaN,NaN,Between Slow & Fast EMA,19,720.89,720.70,757.05,752.17,753.27,Yes,2024-05-07 00:00:00,770.30,0.73,5d
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,TCS,1h,EMA21/EMA50,2026-04-01 13:15:00,2415.70,2406.05,2405.95,2026-04-02 11:15:00,2405.30,2414.70,2403.10,2406.60,2406.59,2406.25,At/Below Slow EMA,5,NaN,NaN,NaN,NaN,2403.10,Yes,2026-04-02 12:15:00,2420.40,0.57,1h
555,TCS,1h,EMA21/EMA50,2026-04-01 13:15:00,2415.70,2406.05,2405.95,2026-04-02 12:15:00,2407.10,2420.40,2403.00,2419.80,2407.79,2406.78,At/Below Slow EMA,6,NaN,NaN,NaN,NaN,2403.00,Yes,2026-04-24 12:15:00,2614.00,8.03,98h
556,TCS,1h,EMA21/EMA50,2026-04-01 13:15:00,2415.70,2406.05,2405.95,2026-04-07 09:15:00,2461.00,2500.00,2448.00,2499.80,2448.15,2428.44,Between Slow & Fast EMA,17,NaN,NaN,NaN,NaN,2448.00,Yes,2026-04-24 10:15:00,2614.00,4.57,85h
557,TCS,1h,EMA21/EMA50,2026-04-01 13:15:00,2415.70,2406.05,2405.95,2026-04-15 09:15:00,2501.90,2553.20,2496.60,2549.30,2510.56,2501.15,At/Below Slow EMA,52,NaN,NaN,NaN,NaN,2496.60,Yes,2026-04-24 09:15:00,2614.00,2.54,49h


In [17]:
# ── SL + TRADE OUTCOME REPORT (A – F) ──────────────────────────────────────

sep = "=" * 70
sw  = signals_with_sl.copy()
sw["Max Upside %"] = pd.to_numeric(sw["Max Upside %"], errors="coerce")
sw["Duration_Num"] = sw["Trade Duration"].str.extract(r"(\d+)").astype(float)

# ── A: SL Hit Rate per Setup ─────────────────────────────────────────────
print(sep)
print("  REPORT A — SL Hit Rate per Setup")
print(sep)
ra = (sw.groupby(["Timeframe", "EMA Pair"])
      .apply(lambda x: pd.Series({
          "Total Signals": len(x),
          "SL Hit":        (x["SL Hit"] == "Yes").sum(),
          "SL Not Hit":    (x["SL Hit"] == "No").sum(),
          "SL Hit Rate %": round((x["SL Hit"] == "Yes").mean() * 100, 1),
      }))
      .reset_index())
display(ra)

# ── B: Max Upside % per Setup ────────────────────────────────────────────
print(sep)
print("  REPORT B — Max Upside % per Setup (Touch Close → highest High)")
print(sep)
rb = (sw.groupby(["Timeframe", "EMA Pair"])
      .agg(Avg_Max_Upside=("Max Upside %", "mean"),
           Median_Upside =("Max Upside %", "median"),
           Max_Upside    =("Max Upside %", "max"),
           Min_Upside    =("Max Upside %", "min"))
      .round(2).reset_index())
display(rb)

# ── C: Trade Duration per Setup ──────────────────────────────────────────
print(sep)
print("  REPORT C — Trade Duration per Setup")
print(sep)
rc = (sw.groupby(["Timeframe", "EMA Pair"])
      .agg(Avg_Duration=("Duration_Num", "mean"),
           Min_Duration=("Duration_Num", "min"),
           Max_Duration=("Duration_Num", "max"))
      .round(1).reset_index())
rc["Unit"] = rc["Timeframe"].apply(lambda x: "hours" if x in ("60", "1h") else "days")
display(rc)

# ── D: Top 10 Best Trades ────────────────────────────────────────────────
print(sep)
print("  REPORT D — Top 10 Best Trades (Max Upside %)")
print(sep)
display(sw.nlargest(10, "Max Upside %")
          [["Stock", "Timeframe", "EMA Pair", "Touch Time", "Touch Close",
            "SL Price", "Max High Before SL", "Max Upside %",
            "SL Hit", "Trade Duration"]])

# ── E: Top 10 Worst Trades ───────────────────────────────────────────────
print(sep)
print("  REPORT E — Top 10 Worst Trades (SL Hit, lowest Max Upside %)")
print(sep)
display(sw[sw["SL Hit"] == "Yes"]
          .nsmallest(10, "Max Upside %")
          [["Stock", "Timeframe", "EMA Pair", "Touch Time", "Touch Close",
            "SL Price", "SL Hit Time", "Max Upside %", "Trade Duration"]])

# ── F: Per Stock Summary ─────────────────────────────────────────────────
print(sep)
print("  REPORT F — Per Stock Summary")
print(sep)
rf = (sw.groupby("Stock")
      .apply(lambda x: pd.Series({
          "Total Signals": len(x),
          "SL Hit Rate %": round((x["SL Hit"] == "Yes").mean() * 100, 1),
          "Avg Upside %":  round(x["Max Upside %"].mean(), 2),
          "Best Upside %": round(x["Max Upside %"].max(), 2),
          "Avg Duration":  round(x["Duration_Num"].mean(), 1),
      }))
      .reset_index())
display(rf)
print(sep)


  REPORT A — SL Hit Rate per Setup


,Timeframe,EMA Pair,Total Signals,SL Hit,SL Not Hit,SL Hit Rate %
0,1d,EMA10/EMA20,229.00,226.00,3.00,98.70
1,1d,EMA21/EMA50,141.00,138.00,3.00,97.90
2,1h,EMA10/EMA20,128.00,123.00,5.00,96.10
3,1h,EMA21/EMA50,61.00,59.00,2.00,96.70


  REPORT B — Max Upside % per Setup (Touch Close → highest High)


,Timeframe,EMA Pair,Avg_Max_Upside,Median_Upside,Max_Upside,Min_Upside
0,1d,EMA10/EMA20,2.76,1.12,47.09,-1.59
1,1d,EMA21/EMA50,4.07,1.42,47.09,-0.45
2,1h,EMA10/EMA20,1.35,0.50,13.67,-1.49
3,1h,EMA21/EMA50,1.43,0.72,8.09,-1.49


  REPORT C — Trade Duration per Setup


...

,Stock,Timeframe,EMA Pair,Touch Time,Touch Close,SL Price,Max High Before SL,Max Upside %,SL Hit,Trade Duration
129,ICICIBANK,1d,EMA10/EMA20,2024-02-13 00:00:00,1019.80,1000.30,1500.00,47.09,No,552d
193,ICICIBANK,1d,EMA21/EMA50,2024-02-13 00:00:00,1019.80,1000.30,1500.00,47.09,No,552d
132,ICICIBANK,1d,EMA10/EMA20,2024-02-29 00:00:00,1052.20,1038.50,1500.00,42.56,No,540d
206,ICICIBANK,1d,EMA21/EMA50,2024-06-18 00:00:00,1122.85,1101.20,1500.00,33.59,No,468d
7,HDFCBANK,1d,EMA10/EMA20,2024-06-05 00:00:00,775.90,741.17,1020.50,31.52,Yes,447d
58,HDFCBANK,1d,EMA21/EMA50,2024-06-05 00:00:00,775.90,741.17,1020.50,31.52,Yes,447d
207,ICICIBANK,1d,EMA21/EMA50,2024-08-22 00:00:00,1191.10,1176.60,1500.00,25.93,No,423d
62,HDFCBANK,1d,EMA21/EMA50,2024-08-16 00:00:00,816.05,805.50,1020.50,25.05,Yes,396d
400,RELIANCE,1d,EMA21/EMA50,2024-01-24 00:00:00,1343.88,1323.92,1608.80,19.71,Yes,186d
371,RELIANCE,1d,EMA10/EMA20,2024-01-25 00:00:00,1353.08,1335.20,1608.80,18.90,Yes,184d


  REPORT E — Top 10 Worst Trades (SL Hit, lowest Max Upside %)


,Stock,Timeframe,EMA Pair,Touch Time,Touch Close,SL Price,SL Hit Time,Max Upside %,Trade Duration
278,INFY,1d,EMA10/EMA20,2024-10-17 00:00:00,1968.10,1930.15,2024-10-18 00:00:00,-1.59,1d
352,INFY,1h,EMA10/EMA20,2026-04-21 15:15:00,1314.60,1311.80,2026-04-22 09:15:00,-1.49,1h
370,INFY,1h,EMA21/EMA50,2026-04-21 15:15:00,1314.60,1311.80,2026-04-22 09:15:00,-1.49,1h
290,INFY,1d,EMA10/EMA20,2025-06-20 00:00:00,1622.90,1608.90,2025-06-23 00:00:00,-1.35,1d
305,INFY,1d,EMA10/EMA20,2026-04-21 00:00:00,1313.20,1299.30,2026-04-22 00:00:00,-1.18,1d
267,INFY,1d,EMA10/EMA20,2024-02-13 00:00:00,1684.55,1663.50,2024-02-14 00:00:00,-0.90,1d
335,INFY,1h,EMA10/EMA20,2026-03-06 15:15:00,1311.00,1306.60,2026-03-09 09:15:00,-0.79,1h
491,TCS,1d,EMA10/EMA20,2025-05-21 00:00:00,3525.80,3498.30,2025-05-22 00:00:00,-0.79,1d
167,ICICIBANK,1d,EMA10/EMA20,2025-04-04 00:00:00,1335.30,1322.10,2025-04-07 00:00:00,-0.77,1d
21,HDFCBANK,1d,EMA10/EMA20,2025-02-19 00:00:00,863.60,856.23,2025-02-20 00:00:00,-0.72,1d


  REPORT F — Per Stock Summary


,Stock,Total Signals,SL Hit Rate %,Avg Upside %,Best Upside %,Avg Duration
0,HDFCBANK,120.00,98.30,2.75,31.52,27.20
1,ICICIBANK,144.00,95.10,3.23,47.09,30.00
2,INFY,107.00,100.00,1.92,12.13,10.10
3,RELIANCE,101.00,96.00,2.58,19.71,16.40
4,TCS,87.00,100.00,2.37,16.73,16.40


In [18]:
# ── EXPORT WITH SL + OUTCOMES ───────────────────────────────────────────────
signals_dir = "signals"
os.makedirs(signals_dir, exist_ok=True)

out = os.path.join(signals_dir, "signals_with_sl.csv")
signals_with_sl.to_csv(out, index=False)
print(f"Saved → {out}")

for (tf, pair), grp in signals_with_sl.groupby(["Timeframe", "EMA Pair"]):
    fname = f"signals_{tf}_{pair.replace('/', '_')}_with_sl.csv"
    fpath = os.path.join(signals_dir, fname)
    grp.to_csv(fpath, index=False)
    print(f"Saved → {fpath}  ({len(grp)} rows)")

print(f"\nAll files saved in → '{signals_dir}/'")


Saved → signals\signals_with_sl.csv
Saved → signals\signals_1d_EMA10_EMA20_with_sl.csv  (229 rows)
Saved → signals\signals_1d_EMA21_EMA50_with_sl.csv  (141 rows)
Saved → signals\signals_1h_EMA10_EMA20_with_sl.csv  (128 rows)
Saved → signals\signals_1h_EMA21_EMA50_with_sl.csv  (61 rows)

All files saved in → 'signals/'


## Target % Analysis

In [19]:
# ── TARGET % ENGINE ────────────────────────────────────────────────────────

def add_target_outcome(signals_sl: pd.DataFrame,
                       stock_data: dict,
                       target_pct: float) -> pd.DataFrame:
    """
    Checks whether price reached target_pct above Touch Close
    before the SL (Touch Low) was hit.

    Adds columns:
      Target Price         = Touch Close * (1 + target_pct/100)
      Target Hit           = Yes / No
      Target Hit Time      = datetime of first breach, else 'Not Hit'
      Target Hit Before SL = Yes / No
      Candles to Target    = candles from touch+1 to target hit
    """
    rows = []
    for _, row in signals_sl.iterrows():
        stock    = row["Stock"]
        interval = row["Timeframe"]
        t_close  = float(row["Touch Close"])
        sl_price = float(row["SL Price"])
        touch_ts = pd.Timestamp(row["Touch Time"])

        target_price = round(t_close * (1 + target_pct / 100), 2)

        df = stock_data.get((stock, interval))
        if df is None:
            rows.append({**row, "Target Price": target_price,
                         "Target Hit": "N/A", "Target Hit Time": None,
                         "Target Hit Before SL": "N/A",
                         "Candles to Target": None})
            continue

        try:
            touch_idx = df.index.get_loc(touch_ts)
        except KeyError:
            touch_idx = df.index.searchsorted(touch_ts)

        target_hit = False; target_hit_time = None
        target_hit_idx = None; sl_hit_idx_local = None

        for j in range(touch_idx + 1, len(df)):
            candle_high = float(df["High"].iloc[j])
            candle_low  = float(df["Low"].iloc[j])

            if not target_hit and candle_high >= target_price:
                target_hit = True
                target_hit_time = df.index[j]
                target_hit_idx  = j

            if candle_low < sl_price:
                sl_hit_idx_local = j
                break

        if target_hit and sl_hit_idx_local is not None:
            hit_before_sl = "Yes" if target_hit_idx <= sl_hit_idx_local else "No"
        elif target_hit:
            hit_before_sl = "Yes"
        else:
            hit_before_sl = "No"

        candles_to_target = (target_hit_idx - touch_idx
                             if target_hit_idx is not None else None)

        rows.append({
            **row,
            "Target Price":         target_price,
            "Target Hit":           "Yes" if target_hit else "No",
            "Target Hit Time":      str(target_hit_time) if target_hit_time else "Not Hit",
            "Target Hit Before SL": hit_before_sl,
            "Candles to Target":    candles_to_target,
        })
    return pd.DataFrame(rows)


print("Target engine loaded.")
print("  Target Price = Touch Close × (1 + target_pct / 100)")
print("  Target checked before SL on every candle (optimistic intra-candle)")


Target engine loaded.
  Target Price = Touch Close × (1 + target_pct / 100)
  Target checked before SL on every candle (optimistic intra-candle)


In [20]:
# ── report_for_target(target_pct, export=False) ────────────────────────────
#
# Call with any target % to get the full analysis instantly.
#
# Examples:
#   report_for_target(3.0)
#   report_for_target(5.0, export=True)
#   for t in [1, 2, 3, 5, 10]: report_for_target(t)

def report_for_target(target_pct: float,
                      export: bool = False) -> pd.DataFrame:
    """
    Run target analysis for the given target_pct and print reports G–J.

    Parameters
    ----------
    target_pct : float  — target return in % above Touch Close (e.g. 3.0 → +3%)
    export     : bool   — if True, saves CSVs to signals/ folder

    Returns
    -------
    pd.DataFrame  — signals_with_sl + target columns
    """
    sep  = "=" * 70
    sep2 = "-" * 70

    sw = add_target_outcome(signals_with_sl, stock_data, target_pct).copy()
    sw["Max Upside %"]      = pd.to_numeric(sw["Max Upside %"],      errors="coerce")
    sw["Candles to Target"] = pd.to_numeric(sw["Candles to Target"], errors="coerce")

    total         = len(sw)
    target_hit_n  = (sw["Target Hit"] == "Yes").sum()
    hit_before_sl = (sw["Target Hit Before SL"] == "Yes").sum()
    sl_only       = ((sw["SL Hit"] == "Yes") &
                     (sw["Target Hit Before SL"] != "Yes")).sum()

    print(sep)
    print(f"  TARGET ANALYSIS  —  {target_pct}% Target")
    print(sep)
    print(f"  Total signals          : {total}")
    print(f"  Target Hit (any time)  : {target_hit_n}  ({target_hit_n/total*100:.1f}%)")
    print(f"  Target Hit Before SL   : {hit_before_sl}  ({hit_before_sl/total*100:.1f}%)")
    print(f"  SL Hit (target missed) : {sl_only}  ({sl_only/total*100:.1f}%)")
    print()

    # ── G: Hit Rate per Setup ────────────────────────────────────────────
    print(sep2)
    print(f"  REPORT G — Target ({target_pct}%) Hit Rate per Setup")
    print(sep2)
    rg = (sw.groupby(["Timeframe", "EMA Pair"])
          .apply(lambda x: pd.Series({
              "Total":               len(x),
              "Target Hit":          (x["Target Hit"] == "Yes").sum(),
              "Hit Before SL":       (x["Target Hit Before SL"] == "Yes").sum(),
              "Target Hit Rate %":   round((x["Target Hit"] == "Yes").mean() * 100, 1),
              "Hit Before SL %":     round((x["Target Hit Before SL"] == "Yes").mean() * 100, 1),
              "SL First (missed) %": round(((x["SL Hit"] == "Yes") &
                                             (x["Target Hit Before SL"] != "Yes")).mean() * 100, 1),
          }))
          .reset_index())
    display(rg)

    # ── H: Speed to Target ───────────────────────────────────────────────
    print(sep2)
    print(f"  REPORT H — Candles to Reach {target_pct}% Target (hits only)")
    print(sep2)
    hits = sw[sw["Target Hit"] == "Yes"]
    if hits.empty:
        print("  No target hits found.")
    else:
        rh = (hits.groupby(["Timeframe", "EMA Pair"])
              .agg(Count      =("Candles to Target", "count"),
                   Avg_Candles=("Candles to Target", "mean"),
                   Min_Candles=("Candles to Target", "min"),
                   Max_Candles=("Candles to Target", "max"))
              .round(1).reset_index())
        display(rh)

    # ── I: Per Stock Summary ─────────────────────────────────────────────
    print(sep2)
    print(f"  REPORT I — Per Stock Target ({target_pct}%) Summary")
    print(sep2)
    ri = (sw.groupby("Stock")
          .apply(lambda x: pd.Series({
              "Total":               len(x),
              "Target Hit Rate %":   round((x["Target Hit"] == "Yes").mean() * 100, 1),
              "Hit Before SL %":     round((x["Target Hit Before SL"] == "Yes").mean() * 100, 1),
              "Avg Candles→Target":  round(x.loc[x["Target Hit"] == "Yes",
                                                   "Candles to Target"].mean(), 1),
              "Avg Max Upside %":    round(x["Max Upside %"].mean(), 2),
          }))
          .reset_index())
    display(ri)

    # ── J: Full table ────────────────────────────────────────────────────
    print(sep2)
    print(f"  REPORT J — Full Signal Table  [{target_pct}% Target]")
    print(sep2)
    display(sw[["Stock", "Timeframe", "EMA Pair", "Touch Time", "Touch Close",
                "SL Price", "SL Hit", "Target Price", "Target Hit",
                "Target Hit Before SL", "Candles to Target",
                "Max High Before SL", "Max Upside %", "Trade Duration"]])
    print(sep)

    # ── Optional export ──────────────────────────────────────────────────
    if export:
        signals_dir = "signals"
        os.makedirs(signals_dir, exist_ok=True)
        tag = str(target_pct).replace(".", "p")
        out = os.path.join(signals_dir, f"signals_target_{tag}pct.csv")
        sw.to_csv(out, index=False)
        print(f"Exported → {out}")
        for (tf, pair), grp in sw.groupby(["Timeframe", "EMA Pair"]):
            fname = f"signals_{tf}_{pair.replace('/','_')}_target_{tag}pct.csv"
            fpath = os.path.join(signals_dir, fname)
            grp.to_csv(fpath, index=False)
            print(f"Exported → {fpath}  ({len(grp)} rows)")

    return sw


print("report_for_target() loaded.")
print("  Usage : report_for_target(3.0)")
print("  Export: report_for_target(3.0, export=True)")
print("  Loop  : for t in [1, 2, 3, 5, 10]: report_for_target(t)")


report_for_target() loaded.
  Usage : report_for_target(3.0)
  Export: report_for_target(3.0, export=True)
  Loop  : for t in [1, 2, 3, 5, 10]: report_for_target(t)


In [21]:
# ── CALL report_for_target ──────────────────────────────────────────────────

# Single target
report_for_target(4.0)

# Multiple targets — uncomment to run
# for target in [1.0, 2.0, 3.0, 5.0, 10.0]:
#     report_for_target(target)

# Save returned dataframe + export CSVs
# df_3pct = report_for_target(3.0, export=True)


  TARGET ANALYSIS  —  4.0% Target
  Total signals          : 559
  Target Hit (any time)  : 114  (20.4%)
  Target Hit Before SL   : 114  (20.4%)
  SL Hit (target missed) : 443  (79.2%)

----------------------------------------------------------------------
  REPORT G — Target (4.0%) Hit Rate per Setup
----------------------------------------------------------------------


,Timeframe,EMA Pair,Total,Target Hit,Hit Before SL,Target Hit Rate %,Hit Before SL %,SL First (missed) %
0,1d,EMA10/EMA20,229.00,51.00,51.00,22.30,22.30,77.30
1,1d,EMA21/EMA50,141.00,42.00,42.00,29.80,29.80,70.20
2,1h,EMA10/EMA20,128.00,14.00,14.00,10.90,10.90,89.10
3,1h,EMA21/EMA50,61.00,7.00,7.00,11.50,11.50,86.90


----------------------------------------------------------------------
  REPORT H — Candles to Reach 4.0% Target (hits only)
----------------------------------------------------------------------


,Timeframe,EMA Pair,Count,Avg_Candles,Min_Candles,Max_Candles
0,1d,EMA10/EMA20,51,7.10,1.00,30.00
1,1d,EMA21/EMA50,42,7.90,1.00,30.00
2,1h,EMA10/EMA20,14,13.60,2.00,35.00
3,1h,EMA21/EMA50,7,11.60,7.00,17.00


----------------------------------------------------------------------
  REPORT I — Per Stock Target (4.0%) Summary
----------------------------------------------------------------------


,Stock,Total,Target Hit Rate %,Hit Before SL %,Avg Candles→Target,Avg Max Upside %
0,HDFCBANK,120.00,20.00,20.00,10.50,2.75
1,ICICIBANK,144.00,17.40,17.40,7.70,3.23
2,INFY,107.00,18.70,18.70,7.20,1.92
3,RELIANCE,101.00,22.80,22.80,9.00,2.58
4,TCS,87.00,25.30,25.30,7.80,2.37


----------------------------------------------------------------------
  REPORT J — Full Signal Table  [4.0% Target]
----------------------------------------------------------------------


,Stock,Timeframe,EMA Pair,Touch Time,Touch Close,SL Price,SL Hit,Target Price,Target Hit,Target Hit Before SL,Candles to Target,Max High Before SL,Max Upside %,Trade Duration
0,HDFCBANK,1d,EMA10/EMA20,2024-03-15 00:00:00,726.33,721.12,Yes,755.38,No,No,NaN,728.00,0.23,1d
1,HDFCBANK,1d,EMA10/EMA20,2024-03-19 00:00:00,724.67,717.75,Yes,753.66,No,No,NaN,725.83,0.16,1d
2,HDFCBANK,1d,EMA10/EMA20,2024-03-21 00:00:00,722.88,719.15,Yes,751.80,No,No,NaN,725.38,0.35,1d
3,HDFCBANK,1d,EMA10/EMA20,2024-04-19 00:00:00,765.65,740.12,Yes,796.28,No,No,NaN,778.70,1.70,13d
4,HDFCBANK,1d,EMA10/EMA20,2024-04-29 00:00:00,764.75,753.27,Yes,795.34,No,No,NaN,770.30,0.73,5d
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,TCS,1h,EMA21/EMA50,2026-04-02 11:15:00,2406.60,2403.10,Yes,2502.86,No,No,NaN,2420.40,0.57,1h
555,TCS,1h,EMA21/EMA50,2026-04-02 12:15:00,2419.80,2403.00,Yes,2516.59,Yes,Yes,12.00,2614.00,8.03,98h
556,TCS,1h,EMA21/EMA50,2026-04-07 09:15:00,2499.80,2448.00,Yes,2599.79,Yes,Yes,17.00,2614.00,4.57,85h
557,TCS,1h,EMA21/EMA50,2026-04-15 09:15:00,2549.30,2496.60,Yes,2651.27,No,No,NaN,2614.00,2.54,49h


,Stock,Timeframe,EMA Pair,Cross Time,Cross Close,EMA21 @ Cross,EMA50 @ Cross,Touch Time,Touch Open,Touch High,Touch Low,Touch Close,EMA21 @ Touch,EMA50 @ Touch,Touch Zone,Candles After Cross,EMA10 @ Cross,EMA20 @ Cross,EMA10 @ Touch,EMA20 @ Touch,SL Price,SL Hit,SL Hit Time,Max High Before SL,Max Upside %,Trade Duration,Target Price,Target Hit,Target Hit Time,Target Hit Before SL,Candles to Target
0,HDFCBANK,1d,EMA10/EMA20,2024-03-13 00:00:00,730.20,NaN,NaN,2024-03-15 00:00:00,725.00,729.65,721.12,726.33,NaN,NaN,Between Slow & Fast EMA,2,720.60,719.68,722.70,721.00,721.12,Yes,2024-03-18 00:00:00,728.00,0.23,1d,755.38,No,Not Hit,No,NaN
1,HDFCBANK,1d,EMA10/EMA20,2024-03-13 00:00:00,730.20,NaN,NaN,2024-03-19 00:00:00,717.75,726.20,717.75,724.67,NaN,NaN,At/Below Slow EMA,4,720.60,719.68,723.11,721.53,717.75,Yes,2024-03-20 00:00:00,725.83,0.16,1d,753.66,No,Not Hit,No,NaN
2,HDFCBANK,1d,EMA10/EMA20,2024-03-13 00:00:00,730.20,NaN,NaN,2024-03-21 00:00:00,721.30,725.62,719.15,722.88,NaN,NaN,At/Below Slow EMA,6,720.60,719.68,721.94,721.14,719.15,Yes,2024-03-22 00:00:00,725.38,0.35,1d,751.80,No,Not Hit,No,NaN
3,HDFCBANK,1d,EMA10/EMA20,2024-03-28 00:00:00,723.95,NaN,NaN,2024-04-19 00:00:00,743.27,767.48,740.12,765.65,NaN,NaN,At/Below Slow EMA,13,720.89,720.70,755.62,748.03,740.12,Yes,2024-05-09 00:00:00,778.70,1.70,13d,796.28,No,Not Hit,No,NaN
4,HDFCBANK,1d,EMA10/EMA20,2024-03-28 00:00:00,723.95,NaN,NaN,2024-04-29 00:00:00,757.50,767.23,753.27,764.75,NaN,NaN,Between Slow & Fast EMA,19,720.89,720.70,757.05,752.17,753.27,Yes,2024-05-07 00:00:00,770.30,0.73,5d,795.34,No,Not Hit,No,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,TCS,1h,EMA21/EMA50,2026-04-01 13:15:00,2415.70,2406.05,2405.95,2026-04-02 11:15:00,2405.30,2414.70,2403.10,2406.60,2406.59,2406.25,At/Below Slow EMA,5,NaN,NaN,NaN,NaN,2403.10,Yes,2026-04-02 12:15:00,2420.40,0.57,1h,2502.86,No,Not Hit,No,NaN
555,TCS,1h,EMA21/EMA50,2026-04-01 13:15:00,2415.70,2406.05,2405.95,2026-04-02 12:15:00,2407.10,2420.40,2403.00,2419.80,2407.79,2406.78,At/Below Slow EMA,6,NaN,NaN,NaN,NaN,2403.00,Yes,2026-04-24 12:15:00,2614.00,8.03,98h,2516.59,Yes,2026-04-07 10:15:00,Yes,12.00
556,TCS,1h,EMA21/EMA50,2026-04-01 13:15:00,2415.70,2406.05,2405.95,2026-04-07 09:15:00,2461.00,2500.00,2448.00,2499.80,2448.15,2428.44,Between Slow & Fast EMA,17,NaN,NaN,NaN,NaN,2448.00,Yes,2026-04-24 10:15:00,2614.00,4.57,85h,2599.79,Yes,2026-04-09 12:15:00,Yes,17.00
557,TCS,1h,EMA21/EMA50,2026-04-01 13:15:00,2415.70,2406.05,2405.95,2026-04-15 09:15:00,2501.90,2553.20,2496.60,2549.30,2510.56,2501.15,At/Below Slow EMA,52,NaN,NaN,NaN,NaN,2496.60,Yes,2026-04-24 09:15:00,2614.00,2.54,49h,2651.27,No,Not Hit,No,NaN
